# Project 3: Moderate — The Full Encoder-Decoder Transformer (Sequence-to-Sequence)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

In [2]:
# Seed
np.random.seed(42)
tf.random.set_seed(42)

units = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
         "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", 
         "seventeen", "eighteen", "nineteen"]
tens = ["", "", "twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]

def num_to_words(n):
    if n<20:
        return units[n]
    elif n<100:
        return tens[n//10] + ("" if n%10 == 0 else " " + units[n%10])
    return ""

In [3]:
# Generate pairs for numbers 0 to 99
src_texts=[]
tgt_texts=[]

for _ in range(3000):
    n = np.random.randint(0,100)
    src_texts.append(str(n))
    tgt_texts.append(f"<start> {num_to_words(n)} <end>")

print("Sample")
for i in range(3):
    print(f"{src_texts[i]} -> {tgt_texts[i]}")

Sample
51 -> <start> fifty one <end>
92 -> <start> ninety two <end>
14 -> <start> fourteen <end>


In [4]:
src_max_len = 4   # digits up to 2 places + padding
tgt_max_len = 6   # "<start> twenty nine <end>" fits comfortably in 6 tokens

# Source Vectorizer (Character level for digits)
src_vec = tf.keras.layers.TextVectorization(
    output_mode="int",
    split="character",
    output_sequence_length=src_max_len
)
src_vec.adapt(src_texts)

# Target Vectorizer (Word level for English words)
tgt_vec = tf.keras.layers.TextVectorization(
    output_mode='int',
    output_sequence_length=tgt_max_len+1
)
tgt_vec.adapt(tgt_texts)

src_vocab_size = len(src_vec.get_vocabulary())
tgt_vocab_size = len(tgt_vec.get_vocabulary())

print("source vocab size (digits): ",src_vocab_size)
print("target vocab size (words): ",tgt_vocab_size)

#vectorize
src_enc = src_vec(src_texts).numpy()
tgt_all = tgt_vec(tgt_texts).numpy()

# Teacher forcing split:
# Decoder Input: "<start> forty two"
# Decoder Target: "forty two <end>"
tgt_in = tgt_all[:, :-1]
tgt_out = tgt_all[:, 1:]

print(f"Shapes -> src: {src_enc.shape}, tgt_in: {tgt_in.shape}, tgt_out: {tgt_out.shape}")

source vocab size (digits):  12
target vocab size (words):  32
Shapes -> src: (3000, 4), tgt_in: (3000, 6), tgt_out: (3000, 6)


In [5]:
class GPTEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(maxlen, embed_dim)
        self.maxlen = maxlen

    def call(self, x):
        length = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

In [6]:
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model // num_heads
        )
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="relu"),
            tf.keras.layers.Dense(d_model)
        ])
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_out = self.mha(query=inputs, value=inputs, key=inputs)
        attn_out = self.dropout1(attn_out, training=training)
        out1 = self.ln1(inputs + attn_out)
        
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        out2 = self.ln2(out1 + ffn_out)
        return out2

In [7]:
class TransformerDecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        # 1. Causal Self-Attention (Target looking at Target)
        self.self_mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model // num_heads
        )
        # 2. Cross-Attention (Query from Decoder, Key/Value from Encoder)
        self.cross_mha = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, 
            key_dim=d_model // num_heads
        )
        # Feed-forward
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="relu"),
            tf.keras.layers.Dense(d_model)
        ])
        
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.ln3 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        
        self.drop1 = tf.keras.layers.Dropout(dropout_rate)
        self.drop2 = tf.keras.layers.Dropout(dropout_rate)
        self.drop3 = tf.keras.layers.Dropout(dropout_rate)

    def call(self, tgt_inputs, enc_outputs, training=False):
        # 1. Causal Self-Attention
        self_attn = self.self_mha(
            query=tgt_inputs, 
            value=tgt_inputs, 
            key=tgt_inputs, 
            use_causal_mask=True
        )
        x = self.ln1(tgt_inputs + self.drop1(self_attn, training=training))
        
        # 2. Cross-Attention (Q = decoder states x, K/V = enc_outputs)
        cross_attn = self.cross_mha(
            query=x, 
            value=enc_outputs, 
            key=enc_outputs
        )
        x = self.ln2(x + self.drop2(cross_attn, training=training))
        
        # 3. FFN
        ffn_out = self.ffn(x)
        x = self.ln3(x + self.drop3(ffn_out, training=training))
        
        return x

In [8]:
embed_dim = 64
num_heads = 4
d_ff = 128

# Inputs
enc_inputs = tf.keras.layers.Input(shape=(src_max_len,), dtype=tf.int32, name="encoder_input")
dec_inputs = tf.keras.layers.Input(shape=(tgt_max_len,), dtype=tf.int32, name="decoder_input")

# Embeddings
enc_emb = GPTEmbedding(maxlen=src_max_len, vocab_size=src_vocab_size, embed_dim=embed_dim)(enc_inputs)
dec_emb = GPTEmbedding(maxlen=tgt_max_len, vocab_size=tgt_vocab_size, embed_dim=embed_dim)(dec_inputs)

# 1. Encoder Stack
enc_out = TransformerEncoderBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)(enc_emb)

# 2. Decoder Stack (receives target embeddings + encoder representations)
dec_out = TransformerDecoderBlock(d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)(dec_emb, enc_out)

# 3. Final vocabulary projection
final_logits = tf.keras.layers.Dense(tgt_vocab_size, activation="softmax")(dec_out)

seq2seq_transformer = tf.keras.Model(
    inputs=[enc_inputs, dec_inputs], 
    outputs=final_logits, 
    name="Full_Transformer"
)

In [9]:
seq2seq_transformer.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.003),
    loss = "sparse_categorical_crossentropy",
    metrics = ['accuracy']
)

In [10]:
seq2seq_transformer.summary()

Model: "Full_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)    │ (None, 4)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_input (InputLayer)    │ (None, 6)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gpt_embedding (GPTEmbedding)  │ (None, 4, 64)             │           1,024 │ encoder_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gpt_embedding_1               │ (None, 6, 64)             │           2,432 │ decoder_input[0][0]        │
│ (GPTEmbedding)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_encoder_block     │ (None, 4, 64)             │          33,472 │ gpt_embedding[0][0]        │
│ (TransformerEncoderBlock)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_decoder_block     │ (None, 6, 64)             │          50,240 │ gpt_embedding_1[0][0],     │
│ (TransformerDecoderBlock)     │                           │                 │ transformer_encoder_block… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_4 (Dense)               │ (None, 6, 32)             │           2,080 │ transformer_decoder_block… │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 89,248 (348.62 KB)

 Trainable params: 89,248 (348.62 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = seq2seq_transformer.fit(
    [src_enc, tgt_in], 
    tgt_out,
    batch_size=64,
    epochs=25,
    verbose=1
)

Epoch 1/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.7648 - loss: 0.9334
Epoch 2/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9800 - loss: 0.0911
Epoch 3/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9990 - loss: 0.0157
Epoch 4/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 1.0000 - loss: 0.0054
Epoch 5/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 0.0031
Epoch 6/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 1.0000 - loss: 0.0022
Epoch 7/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 1.0000 - loss: 0.0016
Epoch 8/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 1.0000 - loss: 0.0013
Epoch 9/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 1.0000 - loss: 0.0010
Epoch 10/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 8.3313e-04
Epoch 11/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 1.0000 - loss: 7.3260e-04
Epoch 12/25
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - 

In [12]:
# Create target vocabulary lookups
tgt_vocab = tgt_vec.get_vocabulary()
word_to_idx = {w: i for i, w in enumerate(tgt_vocab)}
idx_to_word = {i: w for i, w in enumerate(tgt_vocab)}

start_token_idx = word_to_idx["start"]
end_token_idx = word_to_idx["end"]

def translate_number(number_str):
    # 1. Vectorize input number string as characters
    src_input = src_vec([number_str]).numpy() # shape (1, 4)
    
    # 2. Initialize decoder input with '<start>' padded to tgt_max_len
    dec_input = np.zeros((1, tgt_max_len), dtype=np.int32)
    dec_input[0, 0] = start_token_idx
    
    translated_words = []
    
    # 3. Autoregressively generate next token step-by-step
    for i in range(tgt_max_len - 1):
        # Forward pass through model
        preds = seq2seq_transformer.predict([src_input, dec_input], verbose=0)
        
        # Look at the prediction for position i
        next_token_id = int(np.argmax(preds[0, i, :]))
        next_word = idx_to_word.get(next_token_id, "")
        
        if next_word == "end" or next_word == "":
            break
            
        translated_words.append(next_word)
        
        # Feed the predicted token into position i + 1 for the next iteration
        dec_input[0, i + 1] = next_token_id
        
    return " ".join(translated_words)

In [13]:
test_numbers = ["7", "14", "23", "50", "88", "99", "0"]

print("--- Translation Results ---")
for num in test_numbers:
    pred_words = translate_number(num)
    print(f"Input: '{num:>2}'  -->  Predicted English: '{pred_words}'")

--- Translation Results ---
Input: ' 7'  -->  Predicted English: 'seven'
Input: '14'  -->  Predicted English: 'fourteen'
Input: '23'  -->  Predicted English: 'twenty three'
Input: '50'  -->  Predicted English: 'fifty'
Input: '88'  -->  Predicted English: 'eighty eight'
Input: '99'  -->  Predicted English: 'ninety nine'
Input: ' 0'  -->  Predicted English: 'zero'
